# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aniqaatiq842-commits/Flyrank-ML-INTERNSHIP/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

# Capstone: Predicting Future Content Performance Trajectories

## Research Question

**Can historical search, engagement, and content signals predict which pages are likely to experience meaningful future performance decline, and can those predictions be converted into a ranked content-action playbook?**
## Decision Supported

The goal is to help a content team prioritize which pages deserve attention first based on their observed historical search performance and predicted future trajectory.

Rather than attempting to prove that a particular content action causes improved search performance, this project treats the model as a **decision-support and prioritization system**. The output is a ranked set of pages accompanied by suggested actions such as protect, improve, investigate, or monitor.

## Research Objective

The project will:

1. Construct features using information available before the prediction window.
2. Define a future-looking performance outcome.
3. Train and evaluate machine-learning models against a transparent baseline.
4. Use a leakage-safe validation strategy appropriate for future prediction.
5. Compare model performance with the baseline using metrics aligned with ranking and prioritization.
6. Convert model outputs into a ranked action playbook with interpretable reason codes.
7. Clearly distinguish observed evidence from directional recommendations and avoid causal claims.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*


This study uses the FlyRank ML Internship dataset to investigate whether historical search-performance and content characteristics can help identify future content performance trajectories.

The analysis is designed around information that would be available at a defined prediction point. Future-window performance information will be treated as the outcome rather than as an input feature.

The analysis will prioritize safe, aggregated search and content signals and will exclude client names, domains, URLs, private queries, credentials, and other identifying information from public outputs.

Before modeling, the dataset will be audited to determine its available time structure, row grain, performance windows, identifiers, and candidate feature fields.


In [4]:
import pandas as pd
import numpy as np
import os

print("Current working directory:")
print(os.getcwd())

print("\nFiles in current directory:")
print(os.listdir("."))

Current working directory:
/content

Files in current directory:
['.config', 'sample_data']


In [5]:


from pathlib import Path

for base in [Path("/content"), Path("/mnt/data")]:
    if base.exists():
        print(f"\n--- Searching {base} ---")
        for p in base.rglob("*"):
            name = p.name.lower()
            if "flyrank" in name or "content_refresh" in name:
                print(p)


--- Searching /content ---


In [6]:


import importlib.util

packages = ["duckdb", "huggingface_hub", "datasets"]

for package in packages:
    print(f"{package}:",
          "installed" if importlib.util.find_spec(package) else "NOT installed")

duckdb: installed
huggingface_hub: installed
datasets: installed


In [7]:
from huggingface_hub import whoami

try:
    user = whoami()
    print("Logged in to Hugging Face as:", user["name"])
except Exception as e:
    print("Not logged in to Hugging Face.")
    print("Error:", e)

Not logged in to Hugging Face.
Error: Token is required to call the /whoami-v2 endpoint, but no token found. You must provide a token or be logged in to Hugging Face with `hf auth login` or `huggingface_hub.login`. See https://huggingface.co/settings/tokens.


In [8]:
from huggingface_hub import login

login()

In [9]:
import duckdb

con = duckdb.connect()

print("DuckDB version:")
print(con.sql("SELECT version()"))

DuckDB version:
┌─────────────┐
│ "version"() │
│   varchar   │
├─────────────┤
│ v1.3.2      │
└─────────────┘



In [10]:
from huggingface_hub import HfApi

api = HfApi()

results = list(api.list_datasets(search="FlyRank", limit=20))

print("FlyRank datasets found:")
for ds in results:
    print(ds.id)

FlyRank datasets found:
FlyRank/internship-warehouse
FlyRank/internship-lanes
FlyRank/internship-starter


In [11]:
from huggingface_hub import list_repo_files

repo_id = "FlyRank/internship-warehouse"

files = list_repo_files(
    repo_id=repo_id,
    repo_type="dataset"
)

print(f"Total files: {len(files)}\n")

for file in files:
    print(file)

Total files: 24

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily

In [12]:
import os
from huggingface_hub import get_token

hf_token = get_token()

print("Hugging Face token available:", hf_token is not None)

Hugging Face token available: True


In [13]:
import duckdb
import os

con = duckdb.connect()

hf_token = get_token()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{hf_token}'
    );
""")

print("DuckDB Hugging Face authentication configured.")

DuckDB Hugging Face authentication configured.


In [14]:
WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"

test = con.sql(f"""
    SELECT *
    FROM read_parquet('{WAREHOUSE}/dim_content.parquet')
    LIMIT 5
""").df()

display(test)

,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword_e754999ab88dd9f2,url_d6091f18cf628794,22,4,108,2026-05-30,2026-07-01,keyword article,...,3,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15682,2555,NaT,NaT,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword_4329d7aede8e208b,url_3a66d2f2e36823ca,31,6,95,2026-06-12,2026-07-01,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15438,2430,NaT,NaT,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,keyword_9b08047d3d2a0406,url_809eda7a7e20b3b2,22,5,82,2026-05-09,2026-07-01,keyword article,...,4,2026-05-06,gemini-generate-content,gemini-3-flash-preview,16576,2645,NaT,NaT,True,False
3,client_04660893ae39614a,content_019f27f634053ca7,keyword_e7cec7ab1804c1c2,url_5fb42bafc4399861,14,3,92,2026-06-15,2026-06-15,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15457,2522,NaT,NaT,True,False
4,client_04660893ae39614a,content_01efa71faea45dcc,keyword_56b0062a1d8b7524,url_ece0abc3e5fb75f9,24,6,98,2026-05-21,2026-06-01,keyword article,...,4,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15776,2552,NaT,NaT,True,False


In [15]:

print("========== dim_content ==========")

content_schema = con.sql(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        '{WAREHOUSE}/dim_content.parquet'
    )
""").df()

display(content_schema)


print("\n========== fact_content_daily_performance ==========")

daily_schema = con.sql(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        '{WAREHOUSE}/fact_content_daily_performance/month=2026-06/data_0.parquet'
    )
""").df()

display(daily_schema)

========== dim_content ==========


,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None



========== fact_content_daily_performance ==========


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [16]:
print("========== SAMPLE DAILY DATA ==========")

daily_sample = con.sql(f"""
    SELECT *
    FROM read_parquet(
        '{WAREHOUSE}/fact_content_daily_performance/month=2026-06/data_0.parquet'
    )
    LIMIT 10
""").df()

display(daily_sample)

========== SAMPLE DAILY DATA ==========


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-01,client_3ffa76342f366962,content_cde79a1a7432ce40,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
1,2026-06-01,client_3ffa76342f366962,content_bbd33968edccaf24,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
2,2026-06-01,client_3ffa76342f366962,content_d41105eaa19670ea,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
3,2026-06-01,client_3ffa76342f366962,content_902b2d9b3d8a19a2,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
4,2026-06-01,client_3ffa76342f366962,content_e70ae5bf6ab35b59,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
5,2026-06-01,client_3ffa76342f366962,content_1a2c2230fb24d7fb,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
6,2026-06-01,client_3ffa76342f366962,content_d6cacc9a22770146,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
7,2026-06-01,client_3ffa76342f366962,content_753f69a23f02d191,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
8,2026-06-01,client_3ffa76342f366962,content_976210e3749658d8,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
9,2026-06-01,client_3ffa76342f366962,content_02178d51c7038074,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06


In [17]:
print("Daily table columns:")
for i, col in enumerate(daily_sample.columns, 1):
    print(f"{i:02d}. {col}")
    print("dim_content columns:")
for i, col in enumerate(content_schema["column_name"], 1):
    print(f"{i:02d}. {col}")

Daily table columns:
01. report_date
dim_content columns:
02. client_hash_id
dim_content columns:
03. content_hash_id
dim_content columns:
04. client_has_gsc
dim_content columns:
05. client_has_ga4
dim_content columns:
06. gsc_data_available
dim_content columns:
07. ga4_data_available
dim_content columns:
08. gsc_impressions
dim_content columns:
09. gsc_clicks
dim_content columns:
10. gsc_sum_position
dim_content columns:
11. gsc_avg_position
dim_content columns:
12. ga4_pageviews
dim_content columns:
13. ga4_sessions
dim_content columns:
14. ga4_users
dim_content columns:
15. ga4_engaged_sessions
dim_content columns:
16. ga4_total_engagement_sec
dim_content columns:
17. sessions_organic
dim_content columns:
18. sessions_direct
dim_content columns:
19. sessions_referral
dim_content columns:
20. sessions_social
dim_content columns:
21. sessions_paid
dim_content columns:
22. sessions_ai
dim_content columns:
23. ai_chatgpt
dim_content columns:
24. ai_perplexity
dim_content columns:
25. ai

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*


## 3.1 Research Design

This study uses a supervised machine-learning approach to identify content pages that are at risk of meaningful future performance decline.

The analysis follows a temporal prediction design. Historical search, traffic, engagement, demand, and content characteristics are used as predictors, while performance observed in a subsequent future window is used to construct the outcome.

The prediction task is designed as a decision-support problem rather than a causal inference problem. The model is intended to prioritize pages for review and action; it does not establish that any particular content intervention causes a change in search performance.

The overall workflow is:

**Historical data → feature construction → future outcome definition → leakage checks → baseline → machine-learning model → time-aware validation → ranked recommendations.**


## 3.2 Prediction Unit

The primary prediction unit is a **content page at a specific prediction date**.

Each observation represents one content page for one client at a defined prediction point. Historical information available before that date is used to construct the features, while performance occurring after that date is used only to construct the future outcome.

The daily performance table therefore provides the temporal observations needed to transform the raw data into page-level prediction examples.

The anonymized `content_hash_id` and `client_hash_id` identifiers are used internally to maintain page and client boundaries but are not exposed as identifying information in the public analysis.


In [18]:
# Verify the basic prediction identifiers in the daily data

daily_sample[[
    "report_date",
    "client_hash_id",
    "content_hash_id"
]].head(10)

,report_date,client_hash_id,content_hash_id
0,2026-06-01,client_3ffa76342f366962,content_cde79a1a7432ce40
1,2026-06-01,client_3ffa76342f366962,content_bbd33968edccaf24
2,2026-06-01,client_3ffa76342f366962,content_d41105eaa19670ea
3,2026-06-01,client_3ffa76342f366962,content_902b2d9b3d8a19a2
4,2026-06-01,client_3ffa76342f366962,content_e70ae5bf6ab35b59
5,2026-06-01,client_3ffa76342f366962,content_1a2c2230fb24d7fb
6,2026-06-01,client_3ffa76342f366962,content_d6cacc9a22770146
7,2026-06-01,client_3ffa76342f366962,content_753f69a23f02d191
8,2026-06-01,client_3ffa76342f366962,content_976210e3749658d8
9,2026-06-01,client_3ffa76342f366962,content_02178d51c7038074


## 3.3 Prediction Point and Time Windows

The analysis uses a rolling temporal design with a **90-day historical feature window** and a **30-day future outcome window**.

For each content page and prediction date:

* The **historical window** consists of the 90 days immediately preceding the prediction date.
* The **prediction date** marks the boundary between information available for modeling and future information.
* The **future window** consists of the 30 days immediately following the prediction date.
* No observations from the future window are used to construct predictive features.

Conceptually:

**90-day historical window → Prediction Point → 30-day future outcome window**

The historical window captures recent search performance, traffic, engagement, demand, and content characteristics. The future window is reserved exclusively for measuring subsequent performance.

Prediction examples are retained only when sufficient historical and future observations are available. This prevents incomplete windows from being treated as comparable outcomes.


In [19]:
print("=== DAILY TABLE DATE COVERAGE ===")

date_range = con.sql(f"""
    SELECT
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(DISTINCT report_date) AS unique_dates,
        COUNT(*) AS total_rows
    FROM read_parquet(
        '{WAREHOUSE}/fact_content_daily_performance/month=*/data_0.parquet',
        hive_partitioning=true
    )
""").df()

display(date_range)

=== DAILY TABLE DATE COVERAGE ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,first_date,last_date,unique_dates,total_rows
0,2025-01-27,2026-06-30,520,78835655


In [20]:
print("=== MONTHLY COVERAGE ===")

monthly = con.sql(f"""
    SELECT
        month,
        COUNT(*) AS rows,
        COUNT(DISTINCT report_date) AS days,
        COUNT(DISTINCT content_hash_id) AS contents,
        COUNT(DISTINCT client_hash_id) AS clients
    FROM read_parquet(
        '{WAREHOUSE}/fact_content_daily_performance/month=*/data_0.parquet',
        hive_partitioning=true
    )
    GROUP BY month
    ORDER BY month
""").df()

display(monthly)

=== MONTHLY COVERAGE ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,month,rows,days,contents,clients
0,2025-01,1297,5,476,2
1,2025-02,75985,28,5903,3
2,2025-03,167859,31,10374,4
3,2025-04,285114,30,13046,4
4,2025-05,349923,31,14887,4
5,2025-06,329201,30,16399,9
6,2025-07,469794,31,27945,16
7,2025-08,704962,31,37204,15
8,2025-09,845813,30,53127,23
9,2025-10,2165471,31,110339,31


In [21]:
print("=== CALENDAR DATE CONTINUITY ===")

date_continuity = con.sql(f"""
    WITH dates AS (
        SELECT DISTINCT report_date
        FROM read_parquet(
            '{WAREHOUSE}/fact_content_daily_performance/month=*/data_0.parquet',
            hive_partitioning=true
        )
    ),
    date_bounds AS (
        SELECT
            MIN(report_date) AS first_date,
            MAX(report_date) AS last_date
        FROM dates
    ),
    calendar AS (
        SELECT
            unnest(
                generate_series(
                    first_date,
                    last_date,
                    INTERVAL 1 DAY
                )
            ) AS expected_date
        FROM date_bounds
    )
    SELECT
        COUNT(*) AS expected_calendar_days,
        (SELECT COUNT(*) FROM dates) AS observed_dates,
        COUNT(*) - (SELECT COUNT(*) FROM dates) AS missing_dates
    FROM calendar
""").df()

display(date_continuity)

=== CALENDAR DATE CONTINUITY ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,expected_calendar_days,observed_dates,missing_dates
0,520,520,0


In [22]:
print("=== MISSING CALENDAR DATES ===")

missing_dates = con.sql(f"""
    WITH dates AS (
        SELECT DISTINCT report_date
        FROM read_parquet(
            '{WAREHOUSE}/fact_content_daily_performance/month=*/data_0.parquet',
            hive_partitioning=true
        )
    ),
    date_bounds AS (
        SELECT
            MIN(report_date) AS first_date,
            MAX(report_date) AS last_date
        FROM dates
    ),
    calendar AS (
        SELECT
            unnest(
                generate_series(
                    first_date,
                    last_date,
                    INTERVAL 1 DAY
                )
            ) AS expected_date
        FROM date_bounds
    )
    SELECT expected_date
    FROM calendar
    WHERE expected_date NOT IN (SELECT report_date FROM dates)
    ORDER BY expected_date
""").df()

display(missing_dates)


=== MISSING CALENDAR DATES ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,expected_date


## 3.4 Target / Label Definition

The primary prediction target is the future performance trajectory of a content page.

For each prediction point, the target compares impressions observed during the subsequent 30-day outcome window with impressions observed during the immediately preceding 30-day reference window.

The percentage change is defined as:

**Future change = (future 30-day impressions − prior 30-day impressions) / prior 30-day impressions**

Pages with a meaningful decrease in future impressions will be classified as **Declining**, pages with changes within a predefined stability band will be classified as **Stable**, and pages with meaningful positive change will be classified as **Growing**.

A minimum historical activity threshold will be applied before calculating percentage-based change so that extremely small impression counts do not produce unstable or misleading growth rates.

The 90-day historical period preceding the prediction point remains reserved for feature construction. The future 30-day period is used exclusively for target construction and is never provided to the model as an input.

The classification thresholds and minimum activity threshold will be selected using only information available in the training period and documented before final evaluation.


In [25]:
# 3.4A — Aggregate daily performance into monthly prediction snapshots
#
# We first create daily page-level performance and then use DuckDB
# window functions to construct rolling historical and future periods.

daily = con.sql(f"""
    SELECT
        CAST(report_date AS DATE) AS report_date,
        client_hash_id,
        content_hash_id,
        COALESCE(gsc_impressions, 0) AS impressions,
        COALESCE(gsc_clicks, 0) AS clicks,
        COALESCE(ga4_sessions, 0) AS sessions,
        COALESCE(ga4_engaged_sessions, 0) AS engaged_sessions,
        COALESCE(ga4_total_engagement_sec, 0) AS engagement_sec,
        COALESCE(sessions_ai, 0) AS sessions_ai
    FROM read_parquet(
        '{WAREHOUSE}/fact_content_daily_performance/month=*/data_0.parquet',
        hive_partitioning=true
    )
""")

print("Daily relation created successfully.")

Daily relation created successfully.


In [26]:
metrics_check = con.sql("""
    SELECT
        COUNT(*) AS rows,
        SUM(impressions) AS total_impressions,
        SUM(clicks) AS total_clicks,
        SUM(sessions) AS total_sessions,
        SUM(sessions_ai) AS total_ai_sessions
    FROM daily
""").df()

display(metrics_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,total_impressions,total_clicks,total_sessions,total_ai_sessions
0,78835655,1.763520e+09,6440837.0,8313065.0,83618.0


In [27]:
page_daily = con.sql("""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,

        SUM(impressions) AS impressions,
        SUM(clicks) AS clicks,
        SUM(sessions) AS sessions,
        SUM(engaged_sessions) AS engaged_sessions,
        SUM(engagement_sec) AS engagement_sec,
        SUM(sessions_ai) AS sessions_ai

    FROM daily

    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
""")

print("Daily page-level relation created.")


Daily page-level relation created.


In [28]:
test_page = con.sql("""
    SELECT *
    FROM page_daily
    ORDER BY content_hash_id, report_date
    LIMIT 20
""").df()

display(test_page)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,impressions,clicks,sessions,engaged_sessions,engagement_sec,sessions_ai
0,2025-03-29,client_9958f0a7ae1df715,content_000005d4ced12088,6.0,0.0,0.0,0.0,0.0,0.0
1,2025-03-31,client_9958f0a7ae1df715,content_000005d4ced12088,1.0,0.0,0.0,0.0,0.0,0.0
2,2025-04-01,client_9958f0a7ae1df715,content_000005d4ced12088,5.0,0.0,0.0,0.0,0.0,0.0
3,2025-04-02,client_9958f0a7ae1df715,content_000005d4ced12088,6.0,0.0,0.0,0.0,0.0,0.0
4,2025-04-03,client_9958f0a7ae1df715,content_000005d4ced12088,2.0,0.0,0.0,0.0,0.0,0.0
5,2025-04-04,client_9958f0a7ae1df715,content_000005d4ced12088,7.0,1.0,0.0,0.0,0.0,0.0
6,2025-04-05,client_9958f0a7ae1df715,content_000005d4ced12088,2.0,0.0,0.0,0.0,0.0,0.0
7,2025-04-06,client_9958f0a7ae1df715,content_000005d4ced12088,3.0,0.0,0.0,0.0,0.0,0.0
8,2025-04-07,client_9958f0a7ae1df715,content_000005d4ced12088,2.0,0.0,0.0,0.0,0.0,0.0
9,2025-04-08,client_9958f0a7ae1df715,content_000005d4ced12088,7.0,0.0,0.0,0.0,0.0,0.0


In [30]:
import duckdb

con = duckdb.connect()

print("DuckDB connection restored.")
print(con.sql("SELECT version()").fetchone())

DuckDB connection restored.
('v1.3.2',)


In [31]:
WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
print("WAREHOUSE:", WAREHOUSE)

WAREHOUSE: hf://datasets/FlyRank/internship-warehouse


In [32]:
try:
    print(con.sql("SELECT COUNT(*) FROM page_daily").fetchone())
    print("page_daily is available.")
except Exception as e:
    print("page_daily is not available:", e)

page_daily is not available: Invalid Input Error: Python Object "page_daily" of type "DuckDBPyRelation" not suitable for replacement scan.
The object was created by another Connection and can therefore not be used by this Connection.


In [34]:
from huggingface_hub import login

login()

In [35]:
from huggingface_hub import whoami

print("Logged in as:", whoami()["name"])

Logged in as: abcxyz44568


In [36]:
con.execute("""
    CREATE OR REPLACE SECRET hf_token (
        TYPE huggingface,
        PROVIDER credential_chain
    )
""")

print("DuckDB Hugging Face authentication configured.")

DuckDB Hugging Face authentication configured.


In [37]:
test_auth = con.sql(f"""
    SELECT *
    FROM read_parquet(
        '{WAREHOUSE}/fact_content_daily_performance/month=2025-01/data_0.parquet'
    )
    LIMIT 5
""").df()

display(test_auth)

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01


In [38]:
daily = con.sql(f"""
    SELECT
        CAST(report_date AS DATE) AS report_date,
        client_hash_id,
        content_hash_id,
        COALESCE(gsc_impressions, 0) AS impressions,
        COALESCE(gsc_clicks, 0) AS clicks,
        COALESCE(ga4_sessions, 0) AS sessions,
        COALESCE(ga4_engaged_sessions, 0) AS engaged_sessions,
        COALESCE(ga4_total_engagement_sec, 0) AS engagement_sec,
        COALESCE(sessions_ai, 0) AS sessions_ai
    FROM read_parquet(
        '{WAREHOUSE}/fact_content_daily_performance/month=*/data_0.parquet',
        hive_partitioning=true
    )
""")

print("Daily relation restored successfully.")

Daily relation restored successfully.


In [39]:
daily = con.sql(f"""
    SELECT
        CAST(report_date AS DATE) AS report_date,
        client_hash_id,
        content_hash_id,
        COALESCE(gsc_impressions, 0) AS impressions,
        COALESCE(gsc_clicks, 0) AS clicks,
        COALESCE(ga4_sessions, 0) AS sessions,
        COALESCE(ga4_engaged_sessions, 0) AS engaged_sessions,
        COALESCE(ga4_total_engagement_sec, 0) AS engagement_sec,
        COALESCE(sessions_ai, 0) AS sessions_ai
    FROM read_parquet(
        '{WAREHOUSE}/fact_content_daily_performance/month=*/data_0.parquet',
        hive_partitioning=true
    )
""")

print("Daily relation restored.")

Daily relation restored.


In [40]:
page_daily = con.sql("""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,

        SUM(impressions) AS impressions,
        SUM(clicks) AS clicks,
        SUM(sessions) AS sessions,
        SUM(engaged_sessions) AS engaged_sessions,
        SUM(engagement_sec) AS engagement_sec,
        SUM(sessions_ai) AS sessions_ai

    FROM daily

    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
""")

print("page_daily relation restored.")

page_daily relation restored.


In [41]:
display(
    con.sql("""
        SELECT
            COUNT(*) AS rows,
            MIN(report_date) AS first_date,
            MAX(report_date) AS last_date
        FROM page_daily
    """).df()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,first_date,last_date
0,78829265,2025-01-27,2026-06-30


In [42]:
# 3.4E — Identify valid prediction observations
#
# A prediction point must have:
#   1. at least 90 calendar days of historical coverage
#   2. at least 30 calendar days remaining for the future outcome

prediction_dates = con.sql("""
    SELECT DISTINCT report_date AS prediction_date
    FROM page_daily
    WHERE report_date >= DATE '2025-04-28'
      AND report_date <= DATE '2026-05-31'
""")

print("Valid prediction dates:")
display(prediction_dates.limit(10).df())

print(
    con.sql("""
        SELECT COUNT(*) AS prediction_dates
        FROM prediction_dates
    """).df()
)

Valid prediction dates:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,prediction_date
0,2025-05-08
1,2025-05-19
2,2025-05-17
3,2025-05-31
4,2025-05-11
5,2025-05-15
6,2025-05-21
7,2025-05-29
8,2025-05-12
9,2025-05-23


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   prediction_dates
0               399


In [ ]:
prediction_windows = con.sql("""
    WITH prediction_points AS (
        SELECT
            client_hash_id,
            content_hash_id,
            report_date AS prediction_date,

            SUM(impressions) AS prediction_day_impressions

        FROM page_daily

        WHERE report_date >= DATE '2025-04-28'
          AND report_date <= DATE '2026-05-31'

        GROUP BY
            client_hash_id,
            content_hash_id,
            report_date
    )

    SELECT
        p.client_hash_id,
        p.content_hash_id,
        p.prediction_date,

        -- Historical 90-day impressions
        SUM(
            CASE
                WHEN d.report_date >= p.prediction_date - INTERVAL 90 DAY
                 AND d.report_date <  p.prediction_date
                THEN d.impressions
                ELSE 0
            END
        ) AS impressions_90d,

        -- Reference: previous 30 days
        SUM(
            CASE
                WHEN d.report_date >= p.prediction_date - INTERVAL 30 DAY
                 AND d.report_date <  p.prediction_date
                THEN d.impressions
                ELSE 0
            END
        ) AS impressions_prev_30d,

        -- Outcome: next 30 days
        SUM(
            CASE
                WHEN d.report_date >= p.prediction_date
                 AND d.report_date <  p.prediction_date + INTERVAL 30 DAY
                THEN d.impressions
                ELSE 0
            END
        ) AS impressions_future_30d

    FROM prediction_points p

    JOIN page_daily d
      ON d.client_hash_id = p.client_hash_id
     AND d.content_hash_id = p.content_hash_id
     AND d.report_date >= p.prediction_date - INTERVAL 90 DAY
     AND d.report_date <  p.prediction_date + INTERVAL 30 DAY

    GROUP BY
        p.client_hash_id,
        p.content_hash_id,
        p.prediction_date
""")

print("Prediction windows created.")

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
